# FASTQ to counts
This notebook shows how to use basic tools to transform transcriptomics FASTQ files to generate count files which are typically later analyzed in Python/R.
We will download all the FASTQ files,
Perform QC with FastQC,
Trim? Re-QC
Align the reads to the genome with STARsolo to assess data quality,
and align the reads to the transcriptome to generate the counts file. 

Running this notebook locally requires X GB of space (X for FASTQs, Y for STAR genome, Z for transcriptome),
And ABC GB RAM.

We will be using the ... dataset. It has ...

In [1]:
%%bash
set -euo pipefail

download() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local sdrf_url="https://www.ebi.ac.uk/biostudies/files/${accession}/${accession}.sdrf.txt"
    local sdrf_file="${out_dir}/${accession}.sdrf.txt"
    local urls_file="${out_dir}/read_fastq_urls.txt"
    local fastq_dir="${out_dir}/fastq"

    mkdir -p "${fastq_dir}"

    # Download the SDRF sample sheet.
    curl -L "${sdrf_url}" -o "${sdrf_file}"

    # Extract only biological read FASTQs.
    # For 10x scRNA-seq counting, we want R1 + R2 and do not need I1/I2 here.
    tr '\t' '\n' < "${sdrf_file}" \
        | grep -Eo 'ftp://[^[:space:]]+_(R1|R2)_001\.fastq\.gz' \
        | sort -u > "${urls_file}"

    printf "FASTQ URLs to download: %s\n" "$(wc -l < "${urls_file}")"

    # Download R1/R2 FASTQs in parallel.
    xargs -a "${urls_file}" -n 1 -P 4 wget -c -P "${fastq_dir}"
}

download_if_missing() {
    local data_dir="${1:-data}"
    local accession="E-MTAB-13632"
    local out_dir="${data_dir}/${accession}"
    local fastq_dir="${out_dir}/fastq"

    # Treat the download as complete only if we already have biological reads.
    if compgen -G "${fastq_dir}/*_R1_001.fastq.gz" > /dev/null && \
       compgen -G "${fastq_dir}/*_R2_001.fastq.gz" > /dev/null; then
        printf "R1/R2 FASTQs already present in %s\n" "${fastq_dir}"
    else
        download "${data_dir}"
    fi
}

download_if_missing "${1:-data}"

R1/R2 FASTQs already present in data/E-MTAB-13632/fastq


Before alignment, we should inspect raw read quality, adapter content, duplication, and overrepresented sequences. We will use FastQC for per-sample quality control and MultiQC to combine all reports into one place. This helps decide whether trimming is necessary and if the quality is sufficient before generating counts.

In [2]:
%%bash
set -euo pipefail
# Generate a sample sheet for easy downstream processing.

data_dir="data/E-MTAB-13632"
fastq_dir="${data_dir}/fastq"
sample_sheet="${data_dir}/samples.tsv"

find "${fastq_dir}" -maxdepth 1 -type f -name "*_R1_*.fastq.gz" | sort \
| awk 'BEGIN{OFS="\t"; print "sample_id","read1","read2"}
{
    r1=$0
    r2=$0
    gsub("_R1_","_R2_",r2)

    file=$0
    sub(".*/","",file)

    sample=file
    sub("_R1_001.fastq.gz$","",sample)

    print sample, r1, r2
}' > "${sample_sheet}"

column -ts $'\t' "${sample_sheet}" | head -n 10

sample_id        read1                                                    read2
SITTE1_S4_L001   data/E-MTAB-13632/fastq/SITTE1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTE1_S4_L001_R2_001.fastq.gz
SITTF1_S4_L001   data/E-MTAB-13632/fastq/SITTF1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTF1_S4_L001_R2_001.fastq.gz
SITTG1_S4_L001   data/E-MTAB-13632/fastq/SITTG1_S4_L001_R1_001.fastq.gz   data/E-MTAB-13632/fastq/SITTG1_S4_L001_R2_001.fastq.gz
SITTH10_S4_L001  data/E-MTAB-13632/fastq/SITTH10_S4_L001_R1_001.fastq.gz  data/E-MTAB-13632/fastq/SITTH10_S4_L001_R2_001.fastq.gz


We now run FastQC on all raw FASTQ files. The key outputs to inspect are per-base sequence quality, adapter content, sequence duplication, GC distribution, and overrepresented sequences. In RNA-seq, mild duplication is common for highly expressed transcripts, so the most important early decision point is usually whether adapter contamination or poor tail quality warrants trimming.

In [5]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
sample_sheet="${data_dir}/samples.tsv"
qc_dir="${data_dir}/qc/raw_fastqc"
threads=8

mkdir -p "${qc_dir}"

# run FastQC on all FASTQ files in parallel. [is parallel always recommended? maybe we should also show the basics]
# extract columns 2,3 (R1 and R2 FASTQ file paths)
# convert tabs to newlines to get a list of all FASTQ files
tail -n +2 "${sample_sheet}" \
| cut -f2,3 \
| tr '\t' '\n' \
| xargs -n 1 -P "${threads}" -I {} fastqc \
    --threads 1 \
    --outdir "${qc_dir}" \
    "{}"

xargs: warning: options --max-args and --replace/-I/-i are mutually exclusive, ignoring previous --max-args value


application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip
application/gzip


Started analysis of SITTF1_S4_L001_R2_001.fastq.gz
Started analysis of SITTE1_S4_L001_R1_001.fastq.gz
Started analysis of SITTH10_S4_L001_R1_001.fastq.gz
Started analysis of SITTF1_S4_L001_R1_001.fastq.gz
Started analysis of SITTH10_S4_L001_R2_001.fastq.gz
Started analysis of SITTE1_S4_L001_R2_001.fastq.gz
Started analysis of SITTG1_S4_L001_R1_001.fastq.gz
Started analysis of SITTG1_S4_L001_R2_001.fastq.gz
Approx 5% complete for SITTH10_S4_L001_R1_001.fastq.gz
Approx 10% complete for SITTH10_S4_L001_R1_001.fastq.gz
Approx 5% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 15% complete for SITTH10_S4_L001_R1_001.fastq.gz
Approx 20% complete for SITTH10_S4_L001_R1_001.fastq.gz
Approx 5% complete for SITTG1_S4_L001_R1_001.fastq.gz
Approx 25% complete for SITTH10_S4_L001_R1_001.fastq.gz
Approx 10% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 5% complete for SITTF1_S4_L001_R1_001.fastq.gz
Approx 5% complete for SITTE1_S4_L001_R1_001.fastq.gz
Approx 30% complete for SITTH10_S4_L00

Analysis complete for SITTH10_S4_L001_R1_001.fastq.gz


Approx 40% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 20% complete for SITTF1_S4_L001_R1_001.fastq.gz
Approx 45% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 20% complete for SITTE1_S4_L001_R1_001.fastq.gz
Approx 25% complete for SITTG1_S4_L001_R1_001.fastq.gz
Approx 10% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 50% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 25% complete for SITTF1_S4_L001_R1_001.fastq.gz
Approx 55% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 30% complete for SITTG1_S4_L001_R1_001.fastq.gz
Approx 10% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 25% complete for SITTE1_S4_L001_R1_001.fastq.gz
Approx 10% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 60% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 35% complete for SITTG1_S4_L001_R1_001.fastq.gz
Approx 30% complete for SITTF1_S4_L001_R1_001.fastq.gz
Approx 65% complete for SITTH10_S4_L001_R2_001.fastq.gz
Approx 30% complete for SITTE1_S4_L001_R1_001.fastq.gz
Appr

...

Analysis complete for SITTE1_S4_L001_R1_001.fastq.gz


Approx 50% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 40% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 45% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 55% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 45% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 50% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 60% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 50% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 65% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 55% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 55% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 70% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 60% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 75% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 60% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 65% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 80% complete for SITTG1_S4_L001_R2_001.fastq.gz
Approx 65% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 70%

Analysis complete for SITTG1_S4_L001_R2_001.fastq.gz


Approx 85% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 85% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 90% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 90% complete for SITTE1_S4_L001_R2_001.fastq.gz
Approx 95% complete for SITTF1_S4_L001_R2_001.fastq.gz
Approx 95% complete for SITTE1_S4_L001_R2_001.fastq.gz


Analysis complete for SITTF1_S4_L001_R2_001.fastq.gz
Analysis complete for SITTE1_S4_L001_R2_001.fastq.gz


In [1]:
%%bash
set -euo pipefail

data_dir="data/E-MTAB-13632"
raw_fastqc_dir="${data_dir}/qc/raw_fastqc"
multiqc_dir="${data_dir}/qc/raw_multiqc"

mkdir -p "${multiqc_dir}"

multiqc "${raw_fastqc_dir}" \
    --outdir "${multiqc_dir}" \
    --filename "multiqc_raw_reads"


/// ]8;id=171204;https://multiqc.info\MultiQC]8;;\ 🔍 v1.33

       file_search | Search path: /home/dekel/projects/bioinformatics-code-hub/transcriptomics/single-cell/fastq-to-counts/data/E-MTAB-13632/qc/raw_fastqc
         searching | ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0% 0/16  ━━━━━━━━━━━━━━━━━━━━ 100% 16/16  
            fastqc | Found 8 reports
     write_results | Data        : data/E-MTAB-13632/qc/raw_multiqc/multiqc_raw_reads_data
     write_results | Report      : data/E-MTAB-13632/qc/raw_multiqc/multiqc_raw_reads.html
           multiqc | MultiQC complete
